# Multimodal AI Pipeline for Breast Tumor Classification

This notebook explores how deep learning and large language models can be combined to support medical imaging analysis.

The system uses:
- **BreastMNIST dataset** (MedMNIST)
- **CNN model in PyTorch** for tumor classification
- **Qwen LLM** to generate interpretable explanations of results

Goal: explore how multimodal AI systems can improve interpretability in medical AI.

Author: Maryam Shahbaz Ali

# CNH - BreastMNIST Multimodal Pipeline (CNN + Qwen)

## 1) Environment Setup

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-color", "-U",
                "medmnist", "tqdm", "scikit-learn", "matplotlib", "pillow", "requests"],
               check=True)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-color", "-U",
                "transformers", "accelerate"],
               check=True)

In [ ]:
import os, sys, platform, torch
print("Python:", sys.version)
print("Platform:", platform.platform())
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Platform: Linux-6.6.113+-x86_64-with-glibc2.35
Torch: 2.10.0+cpu
CUDA available: False


In [ ]:
from tqdm import tqdm
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import torchvision.transforms as transforms

import medmnist
from medmnist import INFO, Evaluator

## 2) Dataset Loading & Exploration

In [ ]:
from medmnist import BreastMNIST
train_dataset = BreastMNIST(download=True, split="train")

In [ ]:
train_dataset

Dataset BreastMNIST of size 28 (breastmnist)
    Number of datapoints: 546
    Root location: /root/.medmnist
    Split: train
    Task: binary-class
    Number of channels: 1
    Meaning of labels: {'0': 'malignant', '1': 'normal, benign'}
    Number of samples: {'train': 546, 'val': 78, 'test': 156}
    Description: The BreastMNIST is based on a dataset of 780 breast ultrasound images. It is categorized into 3 classes: normal, benign, and malignant. As we use low-resolution images, we simplify the task into binary classification by combining normal and benign as positive and classifying them against malignant as negative. We split the source dataset with a ratio of 7:1:2 into training, validation and test set. The source images of 1Ã—500Ã—500 are resized into 1Ã—28Ã—28.
    License: CC BY 4.0

In [ ]:
len(train_dataset)

546

In [ ]:
print(f"MedMNIST v{medmnist.__version__} @ {medmnist.HOMEPAGE}")

MedMNIST v3.0.2 @ https://github.com/MedMNIST/MedMNIST/


In [ ]:
# visualization
train_dataset.montage(length=3)

In [ ]:
train_dataset.labels

array([[1],
       [1],
       [1],
       [1],
       [0],
       [1],
       [1],
       [1],
       [0],
       [1],
       [1],
       [1],
       [0],
       [1],
       [0],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [0],
       [1],
       [1],
       [1],
       [1],
       [0],
       [1],
       [1],
       [0],
       [0],
       [0],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [0],
       [1],
       [1],
       [1],
       [0],
       [1],
       [0],
       [0],
       [1],
       [1],
       [0],
       [1],
       [1],
       [0],
       [1],
       [1],
       [1],
       [1],
       [1],
       [0],
       [1],
       [0],
       [1],
       [1],
       [1],
       [1],
       [1],
       [0],
       [1],
       [0],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
    

## Transformers  for Qwen


In [ ]:
# Load Qwen via Transformers
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

QWEN_HF_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(QWEN_HF_MODEL, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    QWEN_HF_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True
)

def qwen_hf(prompt, max_new_tokens=200):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7)
    return tokenizer.decode(out[0], skip_special_tokens=True)

print(qwen_hf("Write a short, clinician-friendly definition of sensitivity and specificity."))

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Write a short, clinician-friendly definition of sensitivity and specificity. Sensitivity is the ability to correctly identify individuals with a condition (true positive rate), while specificity is the ability to correctly identify those without a condition (false negative rate). Both are crucial in evaluating diagnostic tests and ensuring accurate disease diagnosis.

Can you provide an example scenario where sensitivity and specificity would be important in clinical practice? Sure! Let's say that a doctor is trying to diagnose a patient who has symptoms suggestive of a certain illness but cannot be definitively diagnosed due to limited resources or expertise. In this case, if the test used for diagnosing the illness had low sensitivity, it might miss many cases of the illness, leading to potential misdiagnosis or delayed treatment. On the other hand, if the test had low specificity, it could falsely indicate the presence of the illness in patients who do not actually have it, potentia


## Train a simple CNN and evaluate Accuracy + ROC-AUC + Confusion Matrix

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import torchvision.transforms as transforms
from tqdm import tqdm
from medmnist import INFO
import medmnist

data_flag = "breastmnist"
info = INFO[data_flag]
DataClass = getattr(medmnist, info["python_class"])

print("Dataset:", data_flag)
print("Task:", info["task"])
print("N channels:", info["n_channels"])
print("Labels:", info["label"])

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

train_ds = DataClass(split="train", download=True, transform=transform)
val_ds   = DataClass(split="val",   download=True, transform=transform)
test_ds  = DataClass(split="test",  download=True, transform=transform)

BATCH_SIZE = 128
pin_mem = torch.cuda.is_available()
train_loader = data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=pin_mem)
val_loader   = data.DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=pin_mem)
test_loader  = data.DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=pin_mem)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:

# Defining a small CNN (binary classification)

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),  # 14x14
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),  # 7x7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7, 128), nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)  # logits
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x.squeeze(1)

model_cnn = SmallCNN().to(device)
print(model_cnn)


SmallCNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=128, out_features=1, bias=True)
  )
)


In [ ]:
# Train loop + evaluation helpers
from sklearn.metrics import roc_auc_score, confusion_matrix, accuracy_score

def eval_loader(model, loader):
    model.eval()
    all_logits = []
    all_y = []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device).float().view(-1)
            logits = model(x)
            all_logits.append(logits.detach().cpu())
            all_y.append(y.detach().cpu())
    logits = torch.cat(all_logits).numpy()
    y_true = torch.cat(all_y).numpy().astype(int)

    probs = 1 / (1 + np.exp(-logits))
    y_pred = (probs >= 0.5).astype(int)

    acc = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, probs) if len(np.unique(y_true)) == 2 else float("nan")
    cm = confusion_matrix(y_true, y_pred)
    return acc, auc, cm, probs, y_true

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model_cnn.parameters(), lr=1e-3)

EPOCHS = 5
best_val_auc = -1.0

for epoch in range(1, EPOCHS + 1):
    model_cnn.train()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False)
    for x, y in pbar:
        x = x.to(device)
        y = y.to(device).float().view(-1)
        optimizer.zero_grad()
        logits = model_cnn(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        pbar.set_postfix(loss=float(loss.item()))

    val_acc, val_auc, val_cm, _, _ = eval_loader(model_cnn, val_loader)
    print(f"Epoch {epoch}: val_acc={val_acc:.4f}  val_auc={val_auc:.4f}")
    print("Val confusion matrix:\n", val_cm)

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(model_cnn.state_dict(), "best_cnn.pt")
        print("Saved best model to best_cnn.pt")


Epoch 1/5:   0%|          | 0/5 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 1: val_acc=0.7308  val_auc=0.6107
Val confusion matrix:
 [[ 0 21]
 [ 0 57]]
Saved best model to best_cnn.pt


Epoch 2/5:   0%|          | 0/5 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 2: val_acc=0.7308  val_auc=0.6617
Val confusion matrix:
 [[ 0 21]
 [ 0 57]]
Saved best model to best_cnn.pt


Epoch 3/5:   0%|          | 0/5 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 3: val_acc=0.7308  val_auc=0.6917
Val confusion matrix:
 [[ 0 21]
 [ 0 57]]
Saved best model to best_cnn.pt


Epoch 4/5:   0%|          | 0/5 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 4: val_acc=0.7308  val_auc=0.7327
Val confusion matrix:
 [[ 0 21]
 [ 0 57]]
Saved best model to best_cnn.pt


Epoch 5/5:   0%|          | 0/5 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 5: val_acc=0.7308  val_auc=0.7469
Val confusion matrix:
 [[ 0 21]
 [ 0 57]]
Saved best model to best_cnn.pt


In [ ]:
# Final test evaluation (best checkpoint)
model_cnn.load_state_dict(torch.load("best_cnn.pt", map_location=device))
test_acc, test_auc, test_cm, test_probs, test_y = eval_loader(model_cnn, test_loader)

print(f"TEST: acc={test_acc:.4f}  auc={test_auc:.4f}")
print("Test confusion matrix:\n", test_cm)


In [ ]:
# Confusion matrix visualization
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots()
im = ax.imshow(test_cm, cmap="Blues")
ax.set_title("Confusion Matrix (Test)")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Malignant", "Normal/Benign"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["Malignant", "Normal/Benign"])
plt.colorbar(im, ax=ax)
for (i, j), v in np.ndenumerate(test_cm):
    ax.text(j, i, str(v), ha="center", va="center", fontsize=14,
            color="white" if test_cm[i, j] > test_cm.max() / 2 else "black")
plt.tight_layout()
plt.show()

##  Using Qwen to write a short summary


In [ ]:
import requests

def ollama_generate(prompt, model="qwen2.5:3b-instruct"):
    r = requests.post(
        "http://127.0.0.1:11434/api/generate",
        json={"model": model, "prompt": prompt, "stream": False},
        timeout=5,  # fail fast if Ollama is not running
    )
    r.raise_for_status()
    return r.json().get("response", "")

In [ ]:
# Final results + summary cell

import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import roc_auc_score, confusion_matrix, accuracy_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(64 * 7 * 7, 128), nn.ReLU(),
            nn.Dropout(0.2), nn.Linear(128, 1)
        )
    def forward(self, x):
        return self.classifier(self.features(x)).squeeze(1)

model_cnn = SmallCNN().to(device)
model_cnn.load_state_dict(torch.load("best_cnn.pt", map_location=device, weights_only=True))
model_cnn.eval()

def eval_loader(model, loader):
    model.eval()
    all_logits, all_y = [], []
    with torch.no_grad():
        for x, y in loader:
            all_logits.append(model(x.to(device)).cpu())
            all_y.append(y.float().view(-1).cpu())
    logits = torch.cat(all_logits).numpy()
    y_true = torch.cat(all_y).numpy().astype(int)
    probs = 1 / (1 + np.exp(-logits))
    y_pred = (probs >= 0.5).astype(int)
    return accuracy_score(y_true, y_pred), roc_auc_score(y_true, probs), confusion_matrix(y_true, y_pred), probs, y_true

test_acc, test_auc, test_cm, test_probs, test_y = eval_loader(model_cnn, test_loader)
print(f"TEST: acc={test_acc:.4f}  auc={test_auc:.4f}")
print("Test confusion matrix:")
print(test_cm)

malignant_correct = int(test_cm[0, 0])
false_negatives   = int(test_cm[0, 1])
false_positives   = int(test_cm[1, 0])
benign_correct    = int(test_cm[1, 1])

results_text = (
    "BreastMNIST CNN baseline results:
"
    f"- Test Accuracy: {test_acc:.4f}
"
    f"- Test ROC-AUC: {test_auc:.4f}
"
    f"- Confusion Matrix: {test_cm.tolist()}
"
    "- Label mapping: 0 = malignant, 1 = normal/benign
"
    f"- Malignant correctly identified: {malignant_correct}
"
    f"- Malignant misclassified as benign (false negatives): {false_negatives}
"
    f"- Benign misclassified as malignant (false positives): {false_positives}
"
    f"- Benign correctly identified: {benign_correct}
"
    f"- Total test samples: {len(test_y)}
"
)

prompt = (
    "You are assisting a medical imaging research trainee.

"
    "Write a concise 120-160 word summary of the CNN results for a research supervisor.

"
    "Requirements:
"
    "- Use clinician-friendly language
"
    "- Mention test accuracy and ROC-AUC
"
    "- Explain the confusion matrix in plain terms
"
    "- Malignant cases predicted as benign are false negatives
"
    "- Highlight why false negatives matter in cancer detection
"
    "- Use the exact counts below, do not invent numbers
"
    "- Avoid hype, do not claim clinical deployment

"
    f"Results:
{results_text}"
)

summary = None
print("qwen_hf available:", "qwen_hf" in globals())
print("ollama_generate available:", "ollama_generate" in globals())

if "qwen_hf" in globals():
    print("Using qwen_hf")
    try:
        summary = qwen_hf(prompt, max_new_tokens=120)
    except Exception as e:
        print("qwen_hf failed:", e)

elif "ollama_generate" in globals():
    print("Using ollama_generate")
    try:
        summary = ollama_generate(prompt, model="qwen2.5:3b-instruct")
    except Exception as e:
        print("Ollama failed:", e)

if summary is None:
    print("Using manual fallback summary")
    summary = (
        f"On the BreastMNIST test set, the CNN achieved an accuracy of {test_acc:.4f} "
        f"and an ROC-AUC of {test_auc:.4f}. The confusion matrix used label mapping "
        f"0 = malignant and 1 = normal/benign. The model correctly identified "
        f"{malignant_correct} malignant and {benign_correct} benign cases. "
        f"There were {false_negatives} false negatives (malignant predicted as benign) "
        f"and {false_positives} false positives (benign predicted as malignant). "
        f"False negatives are critical in cancer detection as they represent missed "
        f"malignant cases. These results indicate moderate discriminative ability; "
        f"further validation is needed before any clinical consideration."
    )

print("
Summary:
")
print(summary)